In [1]:
import os
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

load_dotenv("../.env")
GEMINI_API_KEY= os.getenv("GEMINI_API_KEY")

In [2]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex

llm= GoogleGenAI(model= "models/gemini-2.0-flash", api_key= GEMINI_API_KEY)
embed_model= GoogleGenAIEmbedding(model_name= "models/gemini-embedding-2-preview", api_key= GEMINI_API_KEY)

Settings.llm= llm
Settings.embed_model= embed_model
Settings.chunk_size= 512

In [3]:
documents= SimpleDirectoryReader("../data").load_data()
index= VectorStoreIndex.from_documents(documents)

print(f"Advanced Research Baseline Ready!")

2026-03-20 11:05:54,293 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-03-20 11:05:55,267 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"


Advanced Research Baseline Ready!


In [4]:
query= "What exactly is the DevGuardian MCP Server and what are its core features?"

baseline_engine= index.as_query_engine(streaming= True)
response= baseline_engine.query(query)

print("Baseline Response")
for token in response.response_gen:
    print(token, end="", flush= True)

2026-03-20 11:12:03,509 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-03-20 11:12:03,534 - INFO - AFC is enabled with max remote calls: 10.


Baseline Response


2026-03-20 11:12:04,393 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"


The DevGuardian MCP Server is the core server that communicates with compatible clients, such as Antigravity and Claude Desktop. It exposes multiple tools that allow it to perform tasks such as debugging errors, generating code, reviewing pull requests, and scanning repositories for security vulnerabilities. Its autonomous engineering capability is powered by LangGraph-based agents. It also includes DevOps automation features and a security scanning module that detects potential credential leaks.


In [5]:
from llama_index.core.postprocessor import LLMRerank
from llama_index.core import QueryBundle

reranker= LLMRerank(
    choice_batch_size= 5,
    top_n= 3,
    llm= Settings.llm
)

In [6]:
advanced_engine= index.as_query_engine(
    streaming= True,
    similarity_top_k= 10,
    node_postprocessors= [reranker]
)

In [7]:
print(f"RERANKED RESPONSE (Top-k: 10 -> Rerank: 3)")
response= advanced_engine.query(query)

for token in response.response_gen:
    print(token, end="", flush= True)

RERANKED RESPONSE (Top-k: 10 -> Rerank: 3)


2026-03-20 11:18:50,794 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-03-20 11:18:50,819 - INFO - AFC is enabled with max remote calls: 10.
2026-03-20 11:19:03,935 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-20 11:19:03,990 - INFO - AFC is enabled with max remote calls: 10.
2026-03-20 11:19:05,121 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-20 11:19:05,149 - INFO - AFC is enabled with max remote calls: 10.
2026-03-20 11:19:06,089 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"


The DevGuardian MCP Server is an AI-powered coding assistant designed to automate stages of the software development lifecycle. It understands project architecture before generating or modifying code. Its features include debugging, code generation, testing, security scanning, DevOps automation, GitHub pull request review, and security checks. It uses AI agents and automated testing pipelines to ensure that the generated code is reliable, secure, and production-ready. The server communicates with clients and exposes tools to perform tasks such as debugging errors, generating code, reviewing pull requests, and scanning repositories for security vulnerabilities.


In [16]:
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever

nodes= list(index.docstore.docs.values())

bm25_retriever= BM25Retriever.from_defaults(
    nodes= nodes,
    similarity_top_k= 5
)

2026-03-20 11:33:49,518 - DEBUG - Building index from IDs objects


In [17]:
hybrid_retriever= QueryFusionRetriever(
    [
        index.as_retriever(similarity_top_k= 5),
        bm25_retriever
    ],
    num_queries= 1,
    use_async= True
)

In [19]:
from llama_index.core.query_engine import RetrieverQueryEngine

hybrid_query_engine = RetrieverQueryEngine.from_args(
    retriever=hybrid_retriever,
    node_postprocessors=[reranker],
    streaming=True
)

print("HYBRID SEARCH SYSTEM READY!")

HYBRID SEARCH SYSTEM READY!


In [20]:
response = hybrid_query_engine.query("What are the key takeaways from the DevGuardian docs?")
for token in response.response_gen:
    print(token, end="", flush=True)

2026-03-20 11:38:56,084 - INFO - AFC is enabled with max remote calls: 10.
2026-03-20 11:38:59,422 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-20 11:38:59,443 - INFO - AFC is enabled with max remote calls: 10.
2026-03-20 11:39:00,277 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"


The DevGuardian MCP Server automates critical development tasks by combining multi-agent collaboration, automated testing, and security mechanisms. It reduces development time, improves code quality, and ensures software systems are reliable and secure. The system also includes DevOps automation features like generating Dockerfiles and GitHub Actions workflows, along with a security scanning module to prevent credential leaks. It can operate locally, leveraging cloud-based AI when needed.


In [21]:
from llama_index.core.indices.query.query_transform import HyDEQueryTransform
from llama_index.core.query_engine import TransformQueryEngine

hyde= HyDEQueryTransform(include_original= True)

hyde_query_engine= TransformQueryEngine(hybrid_query_engine, hyde)

print("HYDE ADVANCED ENGINE READY")

HYDE ADVANCED ENGINE READY


In [22]:
test_query= "Tall me about the code generation part."
response= hyde_query_engine.query(test_query)

print(f"Original Query: {test_query}")
print("-" * 30)
for token in response.response_gen:
    print(token, end="", flush= True)

2026-03-20 11:53:22,748 - INFO - AFC is enabled with max remote calls: 10.
2026-03-20 11:53:24,953 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-20 11:53:26,387 - INFO - AFC is enabled with max remote calls: 10.
2026-03-20 11:53:33,221 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-20 11:53:33,252 - INFO - AFC is enabled with max remote calls: 10.


Original Query: Tall me about the code generation part.
------------------------------


2026-03-20 11:53:34,203 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"


The system can analyze project structures and generate code tailored to existing architectures. It utilizes AI agents and automated testing pipelines to ensure that generated code is reliable, secure, and production-ready. Automated testing also ensures that generated code meets quality standards.
